# Query the bioprocess SQLite database

This notebook introduces tables, primary and foreign keys, filtering, joins, grouping, and parameterized SQL queries. Build the database first with `python -m src.data.build_database`.

In [1]:
from pathlib import Path
import sqlite3

import pandas as pd

project_root = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
database_path = project_root / "database" / "bioprocess.db"
connection = sqlite3.connect(database_path)
connection.execute("PRAGMA foreign_keys = ON")

In [ ]:
# project_root
# database_path

WindowsPath('c:/Users/fang/demoApps/Bioprocess_multimodal_intelligence/database/bioprocess.db')

## Inspect the schema

SQLite stores schema metadata in `sqlite_master`. The application uses five domain tables.

In [4]:
pd.read_sql_query(
    "SELECT name, sql FROM sqlite_master WHERE type = 'table' ORDER BY name",
    connection,
)

,name,sql
0,batches,CREATE TABLE batches (\n batch_id TEXT PRIM...
1,images,CREATE TABLE images (\n image_id TEXT PRIMA...
2,outcomes,CREATE TABLE outcomes (\n batch_id TEXT PRI...
3,sensor_data,CREATE TABLE sensor_data (\n measurement_id...
4,text_records,CREATE TABLE text_records (\n text_id TEXT ...


## SELECT, WHERE, and parameterized queries

The `?` placeholder keeps values separate from SQL syntax. This is the correct pattern for values supplied by a user interface.

In [5]:
batch_id = "B023"
b023 = pd.read_sql_query(
    """
    SELECT time_hr, ph, do_pct, glucose_g_l, lactate_g_l,
           viable_cell_density_million_ml, viability_pct, titer_g_l
    FROM sensor_data
    WHERE batch_id = ?
    ORDER BY time_hr
    """,
    connection,
    params=(batch_id,),
)
b023.head()

,time_hr,ph,do_pct,glucose_g_l,lactate_g_l,viable_cell_density_million_ml,viability_pct,titer_g_l
0,0,7.097473,47.833967,5.677955,0.161535,0.605656,98.276778,0.000000
1,4,7.142821,51.382645,5.674994,0.212508,0.423963,98.350201,0.001563
2,8,7.118719,51.436583,5.807891,0.128734,0.449904,98.538938,0.003225
3,12,7.098765,49.553743,5.473546,0.098549,0.527403,98.012704,0.005163
4,16,7.061826,50.282937,5.329473,0.106789,0.643662,97.926161,0.007526


## JOIN metadata to outcomes

The shared `batch_id` key connects the experimental design in `batches` with final results in `outcomes`.

In [6]:
batch_outcomes = pd.read_sql_query(
    """
    SELECT b.batch_id, b.cell_line, b.media_type, b.feed_strategy,
           o.final_titer_g_l, o.final_viability_pct
    FROM batches AS b
    JOIN outcomes AS o ON o.batch_id = b.batch_id
    ORDER BY b.batch_id
    """,
    connection,
)
batch_outcomes.head()

,batch_id,cell_line,media_type,feed_strategy,final_titer_g_l,final_viability_pct
0,B001,CHO-K1,Media_B,early_feed,2.135267,93.783794
1,B002,GS-CHO,Media_B,early_feed,2.592597,93.420258
2,B003,CHO-S,Media_A,standard,2.429592,93.285569
3,B004,CHO-K1,Media_A,standard,3.040842,93.435607
4,B005,GS-CHO,Media_A,standard,2.423172,93.515588


## GROUP BY

Group runs by media type to compare the number of runs and their average final titer.

In [8]:
pd.read_sql_query(
    """
    SELECT b.media_type, COUNT(*) AS batch_count,
           AVG(o.final_titer_g_l) AS mean_final_titer_g_l
    FROM batches AS b
    JOIN outcomes AS o ON o.batch_id = b.batch_id
    GROUP BY b.media_type
    ORDER BY b.media_type
    """,
    connection,
)

,media_type,batch_count,mean_final_titer_g_l
0,Media_A,20,2.578387
1,Media_B,17,2.310179
2,Media_C,13,2.772280


In [9]:
connection.close()